## Stroke Generation Model

### Mixture Density Network

In [ ]:
import torch
import torch.nn as nn
from torch.nn.init import xavier_uniform
import torch.nn.functional as F
from torch.distributions import Categorical, MultivariateNormal

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from typing import Optional, Tuple

In [ ]:

class MixtureDensityNetwork(nn.Module):
    """
    MDN head with full-covariance Gaussian components.
    Outputs: means [B, K, D], lower Cholesky factors [B, K, D, D], mixing logits [B, K].
    """

    def __init__(self, input_dim, output_dim, num_components, hidden_dim=128, min_diag=1e-3):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.num_components = num_components
        self.hidden_dim = hidden_dim
        self.min_diag = min_diag
        self.num_tril = output_dim * (output_dim + 1) // 2

        rows, cols = torch.tril_indices(output_dim, output_dim)
        self.register_buffer("tril_rows", rows, persistent=False)
        self.register_buffer("tril_cols", cols, persistent=False)
        self.register_buffer("diag_mask", rows == cols, persistent=False)

        self.hidden_layer = nn.Linear(input_dim, hidden_dim)
        self.weights_layer = nn.Linear(hidden_dim, num_components)  # mixing logits
        self.means_layer = nn.Linear(hidden_dim, num_components * output_dim)
        self.scale_layer = nn.Linear(hidden_dim, num_components * self.num_tril)

    def set_weights(self, target_samples: torch.Tensor, sigma_weight_scale: float = 0.1):
        """
        Initialization. `target_samples` [N, D] should already be standardized.
        Means are spread over the target range; each Cholesky factor starts as
        diag(std_d) with zero off-diagonal entries.
        """
        nn.init.kaiming_uniform_(self.hidden_layer.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden_layer.bias)

        nn.init.xavier_uniform_(self.weights_layer.weight)
        nn.init.xavier_uniform_(self.means_layer.weight)
        nn.init.xavier_uniform_(self.scale_layer.weight)

        t_min = target_samples.min(dim=0).values  # [D]
        t_max = target_samples.max(dim=0).values  # [D]
        mean_bias = torch.empty(self.num_components, self.output_dim)
        for k in range(self.num_components):
            mean_bias[k] = (t_min + (t_max - t_min) * (k + 0.5) / self.num_components).cpu()

        with torch.no_grad():
            self.means_layer.bias.copy_(mean_bias.reshape(-1))
            self.scale_layer.weight.mul_(sigma_weight_scale)

            # inverse softplus so that diag(L) starts at the per-dimension data std
            data_std = target_samples.std(dim=0).to(self.scale_layer.bias)
            init_diag = (data_std - self.min_diag).clamp_min(1e-3)
            raw_diag = torch.log(torch.expm1(init_diag))
            bias = torch.zeros(self.num_components, self.num_tril).to(self.scale_layer.bias)
            bias[:, self.diag_mask] = raw_diag
            self.scale_layer.bias.copy_(bias.reshape(-1))

    def _build_scale_tril(self, raw: torch.Tensor) -> torch.Tensor:
        """raw [B, K, m] -> lower-triangular L [B, K, D, D] with positive diagonal."""
        batch = raw.size(0)
        dim = self.output_dim
        tril = raw.new_zeros(batch, self.num_components, dim, dim)
        tril[..., self.tril_rows, self.tril_cols] = raw
        diag = F.softplus(tril.diagonal(dim1=-2, dim2=-1)) + self.min_diag
        return tril.tril(-1) + torch.diag_embed(diag)

    def forward(self, x: torch.Tensor):
        hidden = torch.relu(self.hidden_layer(x))
        mix_logits = self.weights_layer(hidden)
        means = self.means_layer(hidden).view(-1, self.num_components, self.output_dim)
        raw = self.scale_layer(hidden).view(-1, self.num_components, self.num_tril)
        return means, self._build_scale_tril(raw), mix_logits

In [ ]:
class GenerativeModel(nn.Module):
    """Generative model of strokes: embeddings -> LSTM -> trunk -> full-covariance MDN."""

    def __init__(self, input_dim, embedding_dim, LSTM_dim, LSTM_count, output_dim,
                 mixture_components, recur_dim=None, min_diag=1e-3):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.embedding_dim = embedding_dim
        self.lstm_dim = LSTM_dim
        self.lstm_count = LSTM_count
        self.mixture_components = mixture_components
        # dimension of the previous-step vector (defaults to the target dimension)
        self.recur_dim = output_dim if recur_dim is None else recur_dim

        self.input_embedding = nn.Linear(input_dim, embedding_dim)
        self.recur_embedding = nn.Linear(self.recur_dim, embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim * 2, hidden_size=LSTM_dim,
                            num_layers=LSTM_count, batch_first=True)
        self.trunk = nn.Linear(LSTM_dim, LSTM_dim)
        self.mdn = MixtureDensityNetwork(LSTM_dim, output_dim,
                                         num_components=mixture_components, min_diag=min_diag)

    def set_weights(self, target_samples, sigma_weight_scale=0.1):
        self.mdn.set_weights(target_samples, sigma_weight_scale)
        for layer in (self.input_embedding, self.recur_embedding, self.trunk):
            nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    def forward(self, x, prev_output=None, state=None, return_state=False):
        """
        Args:
            x: conditioning input [B, input_dim]
            prev_output: previous-step vector [B, recur_dim] (zeros embedding if None)
            state: optional LSTM state (h, c) carried across calls; None resets it
        Returns:
            means [B, K, D], scale_tril [B, K, D, D], mix_logits [B, K] (+ state if requested)
        """
        input_embedded = self.input_embedding(x)
        if prev_output is not None:
            prev_embedded = self.recur_embedding(prev_output)
        else:
            prev_embedded = torch.zeros_like(input_embedded)

        combined = torch.cat((input_embedded, prev_embedded), dim=-1)
        lstm_out, new_state = self.lstm(combined.unsqueeze(1), state)  # sequence length 1
        trunk_out = torch.relu(self.trunk(lstm_out.squeeze(1)))

        means, scale_tril, mix_logits = self.mdn(trunk_out)
        if return_state:
            return means, scale_tril, mix_logits, new_state
        return means, scale_tril, mix_logits

    @torch.no_grad()
    def sample(self, x, prev_output=None, state=None):
        means, scale_tril, mix_logits = self.forward(x, prev_output, state)
        batch_idx = torch.arange(means.size(0), device=means.device)
        comp_idx = Categorical(logits=mix_logits).sample()
        sel_mean = means[batch_idx, comp_idx]          # [B, D]
        sel_tril = scale_tril[batch_idx, comp_idx]     # [B, D, D]
        eps = torch.randn_like(sel_mean)
        return sel_mean + (sel_tril @ eps.unsqueeze(-1)).squeeze(-1)



### Model Architecture

### Dataset

In [ ]:
class StrokeDataset(Dataset):
    """
    A custom dataset class for loading stroke data from a pandas DataFrame
    """
    def __init__(self, data_frame: pd.DataFrame):
        """
        Args:
            data_frame (pd.DataFrame): The DataFrame containing the stroke data.
        """
        self.data_frame = data_frame

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        return {
            'wp': torch.tensor(self.data_frame.iloc[idx]['wp'], dtype=torch.float32),
            'wp_n': torch.tensor(self.data_frame.iloc[idx]['wp_n'], dtype=torch.float32),
            'prev_t0': torch.tensor(self.data_frame.iloc[idx]['prev_t0'], dtype=torch.float32),
            'prev_mu': torch.tensor(self.data_frame.iloc[idx]['prev_mu'], dtype=torch.float32),
            'prev_sigma': torch.tensor(self.data_frame.iloc[idx]['prev_sigma'], dtype=torch.float32),
            'prev_theta': torch.tensor(self.data_frame.iloc[idx]['prev_theta'], dtype=torch.float32),
            'prev_psi': torch.tensor(self.data_frame.iloc[idx]['prev_psi'], dtype=torch.float32),
            't0': torch.tensor(self.data_frame.iloc[idx]['t0'], dtype=torch.float32),
            'mu': torch.tensor(self.data_frame.iloc[idx]['mu'], dtype=torch.float32),
            'sigma': torch.tensor(self.data_frame.iloc[idx]['sigma'], dtype=torch.float32),
            'p_m': torch.tensor(self.data_frame.iloc[idx]['p_m'], dtype=torch.float32),
            'p_e': torch.tensor(self.data_frame.iloc[idx]['p_e'], dtype=torch.float32)
        }

In [ ]:
class StrokeDataLoader(DataLoader):
    """
    A custom DataLoader class for loading stroke data.
    """
    def __init__(self, dataset: StrokeDataset, batch_size: int = 32, shuffle: bool = True, num_workers: int = 0):
        """
        Args:
            dataset (StrokeDataset): The dataset to load.
            batch_size (int): Number of samples per batch.
            shuffle (bool): Whether to shuffle the data at every epoch.
            num_workers (int): How many subprocesses to use for data loading.
        """
        super(StrokeDataLoader, self).__init__(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)

## Iteration

In [ ]:
def duplication_matrix(dim: int, device=None, dtype=None) -> torch.Tensor:
    """
    Duplication matrix Dup such that vec(S) = Dup @ vech(S) for any symmetric S.
    Shape: [dim * dim, dim * (dim + 1) // 2]. Row-major vec, lower-triangle vech.
    """
    rows, cols = torch.tril_indices(dim, dim, device=device)
    n_free = rows.numel()
    dup = torch.zeros(dim * dim, n_free, device=device, dtype=dtype)
    col_idx = torch.arange(n_free, device=device)
    dup[rows * dim + cols, col_idx] = 1.0
    dup[cols * dim + rows, col_idx] = 1.0
    return dup


def gaussian_fisher(sigma: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Explicit Fisher information blocks of N(mu, Sigma).

    Args:
        sigma: covariance matrices, shape [..., D, D].
    Returns:
        fisher_mu:    [..., D, D]   = Sigma^-1
        fisher_sigma: [..., m, m]   = 1/2 Dup^T (Sigma^-1 (x) Sigma^-1) Dup, m = D(D+1)/2
    """
    dim = sigma.size(-1)
    dup = duplication_matrix(dim, sigma.device, sigma.dtype)
    sigma_inv = torch.linalg.inv(sigma)
    kron = torch.einsum("...ij,...kl->...ikjl", sigma_inv, sigma_inv)
    kron = kron.reshape(*sigma.shape[:-2], dim * dim, dim * dim)
    fisher_sigma = 0.5 * dup.T @ kron @ dup
    return sigma_inv, fisher_sigma


def gaussian_fisher_matrix(sigma: torch.Tensor) -> torch.Tensor:
    """Full block-diagonal Fisher matrix over (mu, vech(Sigma)), shape [..., D+m, D+m]."""
    fisher_mu, fisher_sigma = gaussian_fisher(sigma)
    dim, n_free = fisher_mu.size(-1), fisher_sigma.size(-1)
    full = sigma.new_zeros(*sigma.shape[:-2], dim + n_free, dim + n_free)
    full[..., :dim, :dim] = fisher_mu
    full[..., dim:, dim:] = fisher_sigma
    return full


def gaussian_natural_gradient(
    diff: torch.Tensor,
    sigma: torch.Tensor,
    resp: torch.Tensor,
    explicit_fisher: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Complete-data natural gradient of resp * log N(y; mu, Sigma) w.r.t. (mu, vech(Sigma)).

    Args:
        diff:  y - mu, shape [..., D]
        sigma: covariance, shape [..., D, D]
        resp:  responsibilities, shape [...]
        explicit_fisher: if True, build the Fisher blocks explicitly and solve F x = grad
            (O(D^6) per component: use it to validate, not for training).
            If False, use the equivalent closed form.
    Returns:
        nat_mu:    [..., D]
        nat_sigma: [..., m]  (vech coordinates)
    """
    dim = diff.size(-1)
    rows, cols = torch.tril_indices(dim, dim, device=diff.device)
    outer = diff.unsqueeze(-1) * diff.unsqueeze(-2)  # d d^T

    if not explicit_fisher:
        nat_mu = resp.unsqueeze(-1) * diff
        nat_sigma = resp[..., None, None] * (outer - sigma)
        return nat_mu, nat_sigma[..., rows, cols]

    fisher_mu, fisher_sigma = gaussian_fisher(sigma)
    sigma_inv = fisher_mu
    dup = duplication_matrix(dim, diff.device, diff.dtype)

    grad_mu = resp.unsqueeze(-1) * (sigma_inv @ diff.unsqueeze(-1)).squeeze(-1)
    # symmetric matrix derivative of resp * log N w.r.t. Sigma
    grad_mat = 0.5 * resp[..., None, None] * (sigma_inv @ outer @ sigma_inv - sigma_inv)
    grad_vech = torch.einsum("pm,...p->...m", dup, grad_mat.reshape(*sigma.shape[:-2], dim * dim))

    nat_mu = torch.linalg.solve(fisher_mu, grad_mu.unsqueeze(-1)).squeeze(-1)
    nat_sigma = torch.linalg.solve(fisher_sigma, grad_vech.unsqueeze(-1)).squeeze(-1)
    return nat_mu, nat_sigma


def categorical_natural_gradient(
    resp: torch.Tensor, pi: torch.Tensor, pi_floor: float = 1e-4
) -> torch.Tensor:
    """
    Natural gradient of sum_k r_k log pi_k w.r.t. the softmax logits.
    The Fisher matrix diag(pi) - pi pi^T is singular; r / pi solves F x = r - pi, and the
    mean is removed (softmax is invariant to a constant shift of the logits).
    `pi_floor` avoids huge steps when a component has a vanishing weight.
    """
    ratio = resp / pi.clamp_min(pi_floor)
    return ratio - ratio.mean(dim=-1, keepdim=True)

In [ ]:
def ngem_loss(
    model,
    batch,
    device,
    fisher_mode: str = "closed_form",
    natural_mixing: bool = True,
    pi_floor: float = 1e-4,
    return_nll: bool = False,
):
    """
    Multivariate (full covariance) nGEM surrogate loss.

    E-step: responsibilities from the current parameters (no gradient).
    M-step: the complete-data natural gradient w.r.t. the network outputs
            (means, vech(Sigma), mixing logits) is computed with the Fisher blocks of
            each component and injected into backprop through a surrogate loss, so that
            d(loss)/d(output) = -natural_gradient / batch_size.

    Args:
        fisher_mode: "closed_form" (fast) or "explicit" (builds and solves the Fisher
            blocks; use it for validation).
        natural_mixing: natural gradient for the mixing logits (True) or plain gradient
            r - pi (False).
        pi_floor: lower clamp on pi in the categorical natural gradient.
        return_nll: also return the mixture NLL (detached) for monitoring.
    """
    if fisher_mode not in ("closed_form", "explicit"):
        raise ValueError("fisher_mode must be 'closed_form' or 'explicit'")

    wp = batch["wp"].to(device)
    wp_n = batch["wp_n"].to(device)
    prev_t0 = batch["prev_t0"].to(device)
    prev_mu = batch["prev_mu"].to(device)
    prev_sigma = batch["prev_sigma"].to(device)
    prev_theta = batch["prev_theta"].to(device)
    prev_psi = batch["prev_psi"].to(device)
    t0 = batch["t0"].to(device)
    mu = batch["mu"].to(device)
    sigma = batch["sigma"].to(device)

    # target vector [B, D]
    target = torch.cat([t0, mu, sigma, batch["p_m"].to(device), batch["p_e"].to(device)], dim=-1)
    input_data = torch.cat([wp, wp_n], dim=-1)
    prev_input = torch.cat([prev_t0, prev_mu, prev_sigma, prev_theta, prev_psi], dim=-1)

    means, scale_tril, mix_logits = model(input_data, prev_output=prev_input)
    batch_size, dim = target.shape

    # ---- E-step (no gradient) ----
    with torch.no_grad():
        dist = MultivariateNormal(loc=means.detach(), scale_tril=scale_tril.detach())
        log_prob = dist.log_prob(target.unsqueeze(1))                  # [B, K]
        log_joint = F.log_softmax(mix_logits.detach(), dim=-1) + log_prob
        resp = torch.softmax(log_joint, dim=-1)                        # [B, K]
        nll = -torch.logsumexp(log_joint, dim=-1).mean()
        pi = torch.softmax(mix_logits.detach(), dim=-1)

    # ---- M-step: natural gradients w.r.t. the network outputs ----
    diff = (target.unsqueeze(1) - means).detach()                      # [B, K, D]
    cov = scale_tril @ scale_tril.transpose(-1, -2)                    # Sigma = L L^T (with grad)
    rows, cols = torch.tril_indices(dim, dim, device=target.device)
    cov_vech = cov[..., rows, cols]                                    # [B, K, m]

    nat_mu, nat_cov = gaussian_natural_gradient(
        diff, cov.detach(), resp, explicit_fisher=(fisher_mode == "explicit")
    )
    nat_pi = categorical_natural_gradient(resp, pi, pi_floor) if natural_mixing else resp - pi

    # Surrogate: its gradient w.r.t. (means, vech(Sigma), logits) is minus the natural
    # gradient; autograd then applies the Jacobian (chain rule) down to the weights.
    # Note: vech coordinates (lower triangle only) are used on purpose, so that the
    # off-diagonal entries of Sigma are not counted twice.
    surrogate = (
        (means * nat_mu).sum()
        + (cov_vech * nat_cov).sum()
        + (mix_logits * nat_pi).sum()
    )
    loss = -surrogate / batch_size

    if return_nll:
        return loss, nll
    return loss

